# EXP233 · Own Frontier90 notebook

This is the notebook form of our privately built EXP219 model bundle. EXP219 achieved public LB **0.922**; that number belongs to the parent run, not this notebook until it is independently executed and submitted. The paired development result **0.7427291486** is from the reciprocal embryo experiment, not an unbiased estimate for the unknown-embryo selector.

The notebook reads the **current runtime test Zarr volumes**, verifies each weight and source file against the attached artifact manifest, runs the frozen inference code, and creates one root `submission.csv`. Known 6bba movies use source44 weights; other movies use source6bba weights. This one-source routing is a fixed production policy. Its quality on unseen embryo prefixes is unmeasured, so any public score is only a public-track observation; private robustness comes from paired development evidence for the underlying folds and mechanism diversity relative to P26.

Run top to bottom with Internet off and T4 ×2. Code sections preserve EXP219 model and graph logic. They replace `sha(__file__)`, which a notebook has no value for, with the original script's SHA256 and express the same sole-root output check through `os.walk`. A final independent audit checks the output against actual runtime test shapes and graph constraints.

## 1. Locate attached offline sources and verify the complete model/code artifact manifest.

In [ ]:
"""EXP219: frozen90/60 public graph, single predeclared source per runtime movie.
Known embryos use reciprocal clean folds; unseen embryos use source6bba,
the larger source training cohort. No OOF claim for unseen routing.
"""
from pathlib import Path
import csv,hashlib,importlib.metadata,json,os,subprocess,sys,time
from concurrent.futures import ThreadPoolExecutor
COMPETITION='biohub-cell-tracking-during-development'
WORK=Path('/kaggle/working');INPUT=Path('/kaggle/input')
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def locate(owner,slug):
    found=[p for p in (INPUT/'datasets'/owner/slug,INPUT/slug) if p.is_dir()]
    assert len(found)==1,(owner,slug,found)
    return found[0]
SUPPORT=locate('pilkwang','biohub-tracking-support-pack-50ep-v1')
ARTIFACTS=locate('dmitriigluzdov','biohub-exp219-frontier90-models')
manifest=json.loads((ARTIFACTS/'artifact_manifest.json').read_text())
for name,digest in manifest['files'].items():assert sha(ARTIFACTS/name)==digest,name


## 2. Install pinned offline dependencies, import the inference implementation, and preserve the numerical stack.

In [ ]:
numeric=('numpy','scipy','numba','llvmlite','torch')
before={n:importlib.metadata.version(n) for n in numeric}
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-index','--no-deps','--force-reinstall','--find-links',str(SUPPORT/'wheels'),'-r',str(SUPPORT/'requirements-unet-ilp-kaggle-predownload.txt')])
assert before=={n:importlib.metadata.version(n) for n in numeric}
os.environ.update(OMP_NUM_THREADS='4',MKL_NUM_THREADS='4',OPENBLAS_NUM_THREADS='4',PYTHONDONTWRITEBYTECODE='1')
sys.path[:0]=[str(ARTIFACTS/'code'),str(SUPPORT/'repo/src'),str(SUPPORT/'repo/scripts')]
import numpy as np
import torch
from score_exp214_paired import read_graphs,COLUMNS
from biohub_tracking.io import open_dataset


## 3. Discover test movies and their actual image shapes; route each movie to its single frozen source fold and balance GPU lanes.

In [ ]:
TEST=next(p for p in (INPUT/'competitions'/COMPETITION/'test',INPUT/COMPETITION/'test') if p.is_dir())
paths=sorted(TEST.glob('*.zarr'));assert paths
assert len(paths)==len({p.stem for p in paths})
def source_for(name):return '44b6' if name.startswith('6bba_') else '6bba'
shapes={p.stem:tuple(open_dataset(p,require_tracks=False,load_image=False).image_shape) for p in paths}
visible=os.environ.get('CUDA_VISIBLE_DEVICES')
devices=visible.split(',') if visible else [str(i) for i in range(torch.cuda.device_count())]
devices=devices[:min(2,torch.cuda.device_count())];assert devices
lanes=[[] for _ in devices];loads=[0]*len(devices)
for path in sorted(paths,key=lambda p:(-int(np.prod(shapes[p.stem])),p.stem)):
    lane=min(range(len(devices)),key=lambda i:(loads[i],i))
    lanes[lane].append(path.stem);loads[lane]+=int(np.prod(shapes[path.stem]))


## 4. Execute the frozen model and linker per lane. Each subprocess reads runtime test images and writes a per-lane graph receipt.

In [ ]:
def run_lane(index):
    started=time.monotonic();records=[];graphs={}
    for source in ('44b6','6bba'):
        movies=sorted(n for n in lanes[index] if source_for(n)==source)
        if not movies:continue
        bundle={'source_embryo':source,'training_plan_sha256':manifest['training_plan_sha256']}
        for role in ('primary','secondary','center'):
            name=manifest['folds'][source][role];bundle[role]={'path':str(ARTIFACTS/name),'sha256':manifest['files'][name]}
        tag=f'lane{index}_{source}'
        bp=WORK/(tag+'_bundle.json');bp.write_text(json.dumps(bundle))
        mp=WORK/(tag+'_movies.json');mp.write_text(json.dumps(movies))
        out=WORK/tag
        subprocess.check_call([sys.executable,str(ARTIFACTS/'code/run_exp214_public_fold.py'),'--notebook',str(ARTIFACTS/'code/compact47_v20.ipynb'),'--repo',str(SUPPORT/'repo'),'--bundle',str(bp),'--data-dir',str(TEST),'--movies',str(mp),'--output',str(out)],env=dict(os.environ,CUDA_VISIBLE_DEVICES=devices[index]))
        got=read_graphs(out/'submission.csv');assert set(got)==set(movies)
        graphs.update(got);records.append(json.loads((out/'inference_receipt.json').read_text()))
        (out/'submission.csv').rename(out/'predictions.csv')
        for link in (out/'runtime_test').iterdir():
            assert link.is_symlink() and link.resolve().is_relative_to(TEST.resolve());link.unlink()
        (out/'runtime_test').rmdir()
    return graphs,records,{'lane':index,'device':devices[index],'movies':lanes[index],'elapsed_seconds':time.monotonic()-started}


## 5. Combine lane graphs; require exact coverage of every discovered test movie.

In [ ]:
all_graphs={};receipts=[];timings=[]
with ThreadPoolExecutor(max_workers=len(devices)) as pool:
    for graphs,records,timing in pool.map(run_lane,range(len(devices))):
        assert not set(all_graphs)&set(graphs)
        all_graphs.update(graphs);receipts.extend(records);timings.append(timing)
assert set(all_graphs)=={p.stem for p in paths}


## 6. Serialize nodes and edges with in-volume checks into the sole root submission.csv.

In [ ]:
idx=0;selections=[]
with (WORK/'submission.csv').open('w',newline='') as f:
    writer=csv.writer(f);writer.writerow(COLUMNS)
    for path in paths:
        graph=all_graphs[path.stem]
        assert graph['nodes'],('No predicted nodes',path.stem)
        for node,(t,z,y,x) in sorted(graph['nodes'].items()):
            assert all(0<=v<s for v,s in zip((t,z,y,x),shapes[path.stem]))
            writer.writerow([idx,path.stem,'node',int(node),int(t),format(z,'.17g'),format(y,'.17g'),format(x,'.17g'),-1,-1]);idx+=1
        for source,target in sorted(graph['edges']):
            writer.writerow([idx,path.stem,'edge',-1,-1,-1,-1,-1,int(source),int(target)]);idx+=1
        selections.append({'dataset':path.stem,'source':source_for(path.stem),'rule':'reciprocal_known_else_larger_source6bba','oof_claim':None})
assert set(read_graphs(WORK/'submission.csv'))==set(all_graphs)
assert [Path(d)/n for d,_,fs in os.walk(WORK) for n in fs if n=='submission.csv']==[WORK/'submission.csv']


## 7. Write provenance, timing and per-movie routing. The embedded source SHA identifies the parent script, since notebooks have no __file__.

In [ ]:
receipt={'status':'PASS_FULL_INFERENCE','graph_family':'public','runtime_datasets':sorted(all_graphs),'submission_sha256':sha(WORK/'submission.csv'),'rows':idx,'selections':selections,'runtime_shapes':shapes,'artifact_manifest_sha256':sha(ARTIFACTS/'artifact_manifest.json'),'source_sha256':'dcc140873e155862629f6294f5abdb745cb5d36f11b7b76c861e111134d7ba0a','numerical_stack':before,'fold_receipts':receipts,'unknown_embryo_selector_oof':None,'timings':timings,'routing':'one frozen source per movie; no unknown-prefix double inference'}
(WORK/'exp214_runtime_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
print(json.dumps({k:v for k,v in receipt.items() if k!='fold_receipts'}),flush=True)


## Final runtime graph audit

The audit reads current Zarr shapes independently, checks exact columns, finite values, graph topology, bounds, and the single root output. It cannot turn public score into private validation.

In [ ]:
"""Streaming EXP225 graph audit; safe to append to the production source.

This validates the output and its runtime dataset/shape contract. It does not
by itself prove model provenance or that inference produced the CSV.
"""

import csv as _exp225_csv
import hashlib as _exp225_hashlib
import json as _exp225_json
import os as _exp225_os
import operator as _exp225_operator
from decimal import Decimal as _Exp225Decimal, InvalidOperation as _Exp225InvalidOperation
from pathlib import Path as _Exp225Path


EXP225_COLUMNS = [
    "id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"
]


def _exp225_sha256(path):
    digest = _exp225_hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _exp225_integer(value, column, line):
    try:
        # Decimal avoids float rounding of IDs above 2**53 and fractional fields.
        number = _Exp225Decimal(value)
        if not number.is_finite() or number != number.to_integral_value() or "_" in value:
            raise ValueError
        return int(number)
    except (_Exp225InvalidOperation, ValueError, OverflowError) as exc:
        raise ValueError(f"line {line}: {column} must be a finite integer: {value!r}") from exc


def _exp225_rows(path):
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = _exp225_csv.reader(handle, strict=True)
        if next(reader, None) != EXP225_COLUMNS:
            raise ValueError("incorrect CSV columns or order")
        for line, row in enumerate(reader, 2):
            if len(row) != len(EXP225_COLUMNS):
                raise ValueError(f"line {line}: incorrect field count")
            yield line, row


def audit_submission(csv_path, shapes):
    """Validate one CSV against actual dataset -> (T, Z, Y, X) shapes.

    Rows are streamed twice so edges may precede nodes. Memory is proportional
    to nodes and graph degrees; coordinates and CSV rows are never retained.
    Integral decimal spellings such as ``1.0`` are accepted without rounding.
    Raises ValueError on a failed contract and returns JSON-compatible evidence.
    """
    path = _Exp225Path(csv_path)
    if not shapes:
        raise ValueError("runtime dataset shapes must not be empty")
    dimensions = {}
    for dataset, shape in shapes.items():
        if not isinstance(dataset, str) or not dataset:
            raise ValueError("runtime dataset IDs must be nonempty strings")
        try:
            dims = tuple(_exp225_operator.index(value) for value in shape)
        except TypeError as exc:
            raise ValueError(f"{dataset}: shape must contain four positive integers") from exc
        if len(dims) != 4 or any(value <= 0 for value in dims):
            raise ValueError(f"{dataset}: shape must contain four positive integers")
        dimensions[dataset] = dims

    before_sha = _exp225_sha256(path)
    times = {dataset: {} for dataset in dimensions}
    counts = {dataset: {"nodes": 0, "edges": 0} for dataset in dimensions}
    rows = 0
    numeric_indices = (0, 3, 4, 5, 6, 7, 8, 9)
    for line, row in _exp225_rows(path):
        dataset, kind = row[1], row[2]
        if dataset not in dimensions:
            raise ValueError(f"line {line}: unknown runtime dataset {dataset!r}")
        values = {i: _exp225_integer(row[i], EXP225_COLUMNS[i], line) for i in numeric_indices}
        if values[0] != rows:
            raise ValueError(f"line {line}: global id must be contiguous from zero")
        rows += 1
        if kind == "node":
            node_id = values[3]
            if node_id < 0 or node_id in times[dataset]:
                raise ValueError(f"line {line}: negative or duplicate node ID")
            if values[8] != -1 or values[9] != -1:
                raise ValueError(f"line {line}: node sentinel fields must equal -1")
            if any(not 0 <= values[i] < bound for i, bound in zip((4, 5, 6, 7), dimensions[dataset])):
                raise ValueError(f"line {line}: node coordinate out of runtime bounds")
            times[dataset][node_id] = values[4]
            counts[dataset]["nodes"] += 1
        elif kind == "edge":
            if any(values[i] != -1 for i in (3, 4, 5, 6, 7)):
                raise ValueError(f"line {line}: edge sentinel fields must equal -1")
            if values[8] < 0 or values[9] < 0:
                raise ValueError(f"line {line}: negative edge endpoint")
            counts[dataset]["edges"] += 1
        else:
            raise ValueError(f"line {line}: unknown row_type {kind!r}")
    missing = sorted(dataset for dataset, values in counts.items() if values["nodes"] == 0)
    if missing:
        raise ValueError(f"runtime datasets without any nodes: {missing}")

    parents = {dataset: {} for dataset in dimensions}
    out_degree = {dataset: {} for dataset in dimensions}
    for line, row in _exp225_rows(path):
        if row[2] != "edge":
            continue
        dataset = row[1]
        source = _exp225_integer(row[8], "source_id", line)
        target = _exp225_integer(row[9], "target_id", line)
        node_times = times[dataset]
        if source not in node_times or target not in node_times:
            raise ValueError(f"line {line}: orphan edge endpoint")
        if node_times[target] != node_times[source] + 1:
            raise ValueError(f"line {line}: edge must join consecutive frames")
        if target in parents[dataset]:
            if parents[dataset][target] == source:
                raise ValueError(f"line {line}: duplicate edge")
            raise ValueError(f"line {line}: in-degree exceeds one")
        parents[dataset][target] = source
        degree = out_degree[dataset].get(source, 0) + 1
        if degree > 2:
            raise ValueError(f"line {line}: out-degree exceeds two")
        out_degree[dataset][source] = degree

    after_sha = _exp225_sha256(path)
    if before_sha != after_sha:
        raise ValueError("submission changed while being audited")
    per_dataset = {}
    for dataset in sorted(dimensions):
        node_times, incoming, outgoing = times[dataset], parents[dataset], out_degree[dataset]
        per_dataset[dataset] = {
            **counts[dataset],
            "shape_tzyx": list(dimensions[dataset]),
            "minimum_t": min(node_times.values()),
            "maximum_t": max(node_times.values()),
            "frames_with_nodes": len(set(node_times.values())),
            "maximum_in_degree": 1 if incoming else 0,
            "maximum_out_degree": max(outgoing.values(), default=0),
            "roots": len(node_times) - len(incoming),
            "leaves": len(node_times) - len(outgoing),
            "isolated_nodes": sum(node not in incoming and node not in outgoing for node in node_times),
            "division_nodes": sum(degree == 2 for degree in outgoing.values()),
        }
    return {
        "status": "PASS_EXP225_OUTPUT_AUDIT",
        "submission_sha256": after_sha,
        "submission_bytes": path.stat().st_size,
        "columns": list(EXP225_COLUMNS),
        "rows": rows,
        "nodes": sum(value["nodes"] for value in counts.values()),
        "edges": sum(value["edges"] for value in counts.values()),
        "datasets": sorted(dimensions),
        "runtime_shapes": {dataset: list(dimensions[dataset]) for dataset in sorted(dimensions)},
        "per_dataset": per_dataset,
        "dynamic_runtime_id_equality": True,
        "finite_integer_fields": True,
        "graph_invariants_valid": True,
    }


def run_exp225_output_audit(TEST_DIR, WORKING_DIR, source_evidence_status=None):
    """Read actual runtime Zarr shapes and write the root output audit receipt."""
    import zarr

    test_dir = _Exp225Path(TEST_DIR).resolve()
    working_dir = _Exp225Path(WORKING_DIR).resolve()
    submission = working_dir / "submission.csv"
    candidates = []
    # Filename inventory is deliberately not a glob or an input CSV search.
    for directory, _, filenames in _exp225_os.walk(working_dir):
        if "submission.csv" in filenames:
            candidates.append(_Exp225Path(directory) / "submission.csv")
    if candidates != [submission] or submission.resolve().parent != working_dir:
        raise ValueError(f"expected exactly one root submission.csv: {candidates}")
    runtime_paths = sorted(path for path in test_dir.iterdir() if path.is_dir() and path.suffix == ".zarr")
    if not runtime_paths:
        raise ValueError("runtime test directory has no Zarr datasets")
    shapes = {
        path.stem: tuple(zarr.open_group(str(path), mode="r")["0"].shape)
        for path in runtime_paths
    }
    receipt = audit_submission(submission, shapes)
    receipt.update({
        "status": "PASS_EXP225_RUNTIME_OUTPUT_AUDIT",
        "test_dir": str(test_dir),
        "working_dir": str(working_dir),
        "runtime_datasets": sorted(shapes),
        "runtime_test_paths": [str(path) for path in runtime_paths],
        "runtime_shape_discovery": "zarr.open_group(runtime_test_path, mode='r')['0'].shape",
        "one_root_submission": True,
        "resolved_biohub_environment": {
            key: value for key, value in sorted(_exp225_os.environ.items()) if key.startswith("BIOHUB_")
        },
        "source_evidence_status": source_evidence_status or _exp225_os.environ.get(
            "BIOHUB_SOURCE_EVIDENCE_STATUS", "NOT_PROVIDED"
        ),
        "evidence_scope": "Runtime output/schema/graph audit; inference and source provenance require separate evidence.",
    })
    destination = working_dir / "exp225_runtime_audit.json"
    destination.write_text(_exp225_json.dumps(receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(_exp225_json.dumps({"exp225_runtime_audit": str(destination), "rows": receipt["rows"],
                             "sha256": receipt["submission_sha256"]}))
    return receipt


In [ ]:
EXP233_OUTPUT_AUDIT = run_exp225_output_audit(
    TEST, WORK,
    source_evidence_status='EXP219 owned reciprocal-fold weights and verified manifest; no unseen-selector OOF',
)
print(json.dumps(EXP233_OUTPUT_AUDIT, indent=2, sort_keys=True))
